# 🔍 Basic RAG Pipeline — LangChain + Ollama + ChromaDB

---

## 🧠 What is RAG?

> **RAG = Retrieval-Augmented Generation**

**Analogy:**  
Imagine you're taking an **open-book exam**.
- Without RAG → You rely only on your memory (LLM's training data). Can hallucinate.
- With RAG → You look up the textbook first, then answer. Grounded in facts.

**The 3-step RAG idea:**
```
User Question
     │
     ▼
[ RETRIEVE ] ──→ Search your document store for relevant chunks
     │
     ▼
[ AUGMENT  ] ──→ Inject those chunks into the LLM prompt as context
     │
     ▼
[ GENERATE ] ──→ LLM answers using ONLY that context
```

---

## 🗺️ Full Pipeline Overview

```
┌─────────────────────────────────────────────────────────┐
│                  INDEXING PHASE (run once)              │
│                                                         │
│  Raw Text File                                          │
│       │                                                 │
│  [Step 1] TextLoader   ──→  Load as Document objects   │
│       │                                                 │
│  [Step 2] TextSplitter ──→  Split into small chunks    │
│       │                                                 │
│  [Step 3] OllamaEmbeddings ──→ Vectorize each chunk   │
│       │                                                 │
│  [Step 4] ChromaDB     ──→  Persist vectors to disk    │
└─────────────────────────────────────────────────────────┘
              ↓  (query time)
┌─────────────────────────────────────────────────────────┐
│                  QUERYING PHASE (per question)          │
│                                                         │
│  User Question                                          │
│       │                                                 │
│  [Step 5] Retriever    ──→  Find top-K similar chunks  │
│       │                                                 │
│  [Step 6] Prompt Build ──→  Inject context + question  │
│       │                                                 │
│  [Step 7] ChatOllama   ──→  Generate grounded answer   │
└─────────────────────────────────────────────────────────┘
```

## 📦 Prerequisites & Installation

Before running this notebook, make sure:

1. **Ollama is installed and running** → https://ollama.com  
2. **Models are pulled:**
   ```bash
   ollama pull nomic-embed-text    # embedding model
   ollama pull gpt-oss:120b-cloud  # or any chat model you prefer
   ```
3. **Python packages installed:**
   ```bash
   pip install langchain langchain-community langchain-ollama langchain-chroma chromadb
   ```
4. **A sample document exists at** `docs/company_policy.txt`  
   *(Create the folder and drop any plain text file in there)*

---
## 📚 Step 0 — Imports

We import exactly what we need — one tool per job:

| Import | Role |
|---|---|
| `TextLoader` | Reads a `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors (numbers) |
| `Chroma` | Vector database — stores & searches embeddings |
| `ChatOllama` | The LLM that generates the final answer |

In [1]:
# ── Document Ingestion ──────────────────────────────────────────────────────
from langchain_community.document_loaders.text import TextLoader

# ── Chunking ────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Embeddings (local, via Ollama) ───────────────────────────────────────────
from langchain_ollama import OllamaEmbeddings

# ── Vector Store (local, persisted to disk) ──────────────────────────────────
from langchain_chroma import Chroma

# ── LLM (local chat model via Ollama) ────────────────────────────────────────
from langchain_ollama import ChatOllama

print("✅ All imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_38056\4043076050.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader
d:\RAG Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful


---
## 📄 Step 1 — Load the Document

**What happens here:**  
`TextLoader` reads the raw `.txt` file and wraps it in LangChain's `Document` object — which carries both the `page_content` (the text) and `metadata` (source path, etc.).

**Why `encoding='utf-8'`?**  
Prevents crashes on special characters (accents, symbols). Always specify this in production.

> 💡 **Production Note:** For large corpora, swap `TextLoader` with `DirectoryLoader` to bulk-load hundreds of files.

In [2]:
print("Loading document...\n")

loader = TextLoader(
    "docs/company_policy.txt",
    encoding="utf-8"   # Always explicit — avoids silent data corruption
)

documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview (first 300 chars):")
print(documents[0].page_content[:300])
print(f"\n🏷️  Metadata: {documents[0].metadata}")

Loading document...

✅ Loaded 1 document(s)

📄 Preview (first 300 chars):
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

🏷️  Metadata: {'source': 'docs/company_policy.txt'}


---
## ✂️ Step 2 — Split Document into Chunks

**Why split at all?**  
LLMs have a **context window limit**. You can't feed a 50-page PDF into every prompt. More importantly, smaller chunks = more *precise* retrieval.

**Analogy:**  
Think of a book index. You don't retrieve the entire chapter — you retrieve the exact page that answers the question.

**Key parameters explained:**

| Parameter | Value | Meaning |
|---|---|---|
| `chunk_size` | 200 | Max characters per chunk |
| `chunk_overlap` | 50 | Characters shared between adjacent chunks |

**Why overlap?**  
If a sentence spans a chunk boundary, overlap ensures neither chunk loses the full context of that idea.

```
Chunk 1: [........................................]
                               overlap↓
Chunk 2:                  [........................................]
```

> ⚠️ **Common Mistake:** chunk_size too large → retrieval becomes imprecise (like returning a whole chapter instead of a paragraph). Too small → sentences get cut mid-thought.

In [3]:
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,    # Characters per chunk — tune based on your document density
    chunk_overlap=50   # Overlap prevents losing context at chunk boundaries
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks Created: {len(chunks)}")
print(f"\n--- Previewing first 3 chunks ---")

for index, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {index + 1} ({len(chunk.page_content)} chars)")
    print(chunk.page_content)
    print("-" * 50)

Splitting document into chunks...

✅ Total Chunks Created: 1

--- Previewing first 3 chunks ---

🔹 Chunk 1 (191 chars)
Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
--------------------------------------------------


---
## 🔢 Step 3 — Create the Embedding Model

**What are embeddings?**  
Embeddings transform text into a list of numbers (a vector) that captures *semantic meaning*.

**Analogy:**  
Plot cities on a map using coordinates (lat, long). Cities that are *geographically close* appear near each other.  
Embeddings do the same for *meaning* — "vacation" and "holiday" will be close in vector space.

```
"Leave policy"   → [0.12, -0.45, 0.87, ...] (384 numbers)
"Annual leave"   → [0.14, -0.43, 0.85, ...] (very similar!)
"Server firewall"→ [0.91,  0.23, -0.10, ...] (very different)
```

**Why `nomic-embed-text`?**  
It's a high-quality, open-source embedding model that runs locally via Ollama — no API key, no cost, no data leaving your machine.

> 💡 **Production Note:** For multi-language docs, consider `mxbai-embed-large` or OpenAI's `text-embedding-3-small`.

In [4]:
print("Loading Embedding Model...\n")

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"  # Runs locally via Ollama — no external API calls
)

print("✅ Embedding Model Ready")

# ── Quick sanity check — embed a test sentence ───────────────────────────────
test_vector = embeddings.embed_query("What is the leave policy?")
print(f"\n🔢 Vector dimension: {len(test_vector)}")
print(f"📊 First 5 values: {[round(v, 4) for v in test_vector[:5]]}")

Loading Embedding Model...

✅ Embedding Model Ready

🔢 Vector dimension: 768
📊 First 5 values: [0.0089, 0.0261, -0.1395, -0.0419, 0.051]


---
## 🗄️ Step 4 — Create & Persist the Vector Store (ChromaDB)

**What is ChromaDB?**  
A lightweight, local vector database. It stores your text chunks alongside their embeddings and supports fast **similarity search**.

**Analogy:**  
Think of ChromaDB as a highly optimized filing cabinet where instead of alphabetical tabs, folders are arranged by *topic similarity*. Ask a question and it instantly opens the most relevant drawer.

**What `from_documents()` does internally:**
```
For each chunk:
  1. Call embeddings.embed_query(chunk.page_content)
  2. Store (chunk_text, vector, metadata) in Chroma
  3. Persist to disk at ./chroma_db
```

> ⚠️ **Edge Case:** If you re-run this cell without deleting `./chroma_db`, you may get duplicate entries. In production, use `Chroma(persist_directory=...)` to *load* an existing store instead of recreating it.

In [5]:
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,          # Your text chunks
    embedding=embeddings,      # Embedding model to vectorize them
    persist_directory="./chroma_db"  # Saves to disk — survives kernel restarts
)

print("✅ Vector Store Created & Persisted to ./chroma_db")
print(f"📦 Total vectors stored: {vectorstore._collection.count()}")

Creating Chroma Vector Store...

✅ Vector Store Created & Persisted to ./chroma_db
📦 Total vectors stored: 1


---
## 🔎 Step 5 — Create the Retriever

**What is a Retriever?**  
A retriever is a wrapper around the vector store that handles the *search* logic. You give it a question; it returns the top-K most semantically similar chunks.

**How similarity search works:**
```
1. Embed the user's question → question_vector
2. Compute cosine similarity between question_vector and ALL stored vectors
3. Return top-K chunks with highest similarity scores
```

**`k=2` means:** return the 2 most relevant chunks per query.

> 💡 **Tuning Tip:** Higher `k` = more context for the LLM, but also more noise. Start at 3–5 for most use cases. Use `search_type="mmr"` (Maximal Marginal Relevance) to avoid retrieving near-duplicate chunks.

In [6]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}   # Retrieve top-2 most relevant chunks per query
    # For production, consider: search_type="mmr" to avoid duplicate chunks
)

print("✅ Retriever Ready")

# ── Quick test — see what the retriever returns for a sample question ─────────
test_query = "What is the leave policy?"
test_results = retriever.invoke(test_query)

print(f"\n🔍 Test Query: '{test_query}'")
print(f"📄 Retrieved {len(test_results)} chunk(s):")
for i, doc in enumerate(test_results):
    print(f"\n  Chunk {i+1}: {doc.page_content[:150]}...")

✅ Retriever Ready

🔍 Test Query: 'What is the leave policy?'
📄 Retrieved 1 chunk(s):

  Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days....


---
## 🤖 Step 6 — Load the LLM

**Role of the LLM in RAG:**  
The LLM is the *synthesizer*, not the *knowledge source*. It receives the retrieved chunks as context and generates a fluent, coherent answer. The knowledge comes from your documents — not the model's weights.

**Key parameters:**

| Parameter | Value | Why |
|---|---|---|
| `model` | `gpt-oss:120b-cloud` | Your Ollama model name — swap as needed |
| `temperature` | `0` | Deterministic output — critical for factual Q&A (no creativity) |

> ⚠️ **Always set `temperature=0` for RAG.** Higher temperatures introduce randomness which can cause the LLM to deviate from the provided context — the opposite of what RAG is designed to prevent.

In [7]:
print("Loading LLM...\n")

llm = ChatOllama(
    model="gpt-oss:120b-cloud",  # Replace with your pulled Ollama model
    temperature=0                 # IMPORTANT: 0 = deterministic, no hallucination drift
)

print("✅ LLM Ready")

Loading LLM...

✅ LLM Ready


---
## 💬 Step 7 — Interactive Q&A Loop

**The full RAG flow in action.** For each question:

```
Question → Retriever → Top-K Chunks
                              │
                    Build Prompt (context + question)
                              │
                       LLM generates answer
                              │
                     Print answer to user
```

**Prompt Design — Why this structure?**

The prompt uses a strict instruction: *"Answer ONLY using the provided context."*  
This is the key guardrail that prevents the LLM from using its internal training knowledge (which could be wrong, outdated, or hallucinated).

The fallback — *"I could not find that information in the document"* — is equally important: it prevents confident wrong answers.

> 💡 **Production Enhancement:** Replace this manual loop with `RetrievalQA` or `ConversationalRetrievalChain` from LangChain for built-in conversation memory and cleaner abstractions.

In [18]:
print("RAG Q&A System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:

    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    # ── RETRIEVE ─────────────────────────────────────────────────────────────
    # Embed the question and find the top-K most similar chunks from ChromaDB
    retrieved_docs = retriever.invoke(question)

    print("\n===== 📚 RETRIEVED CHUNKS =====")
    for index, doc in enumerate(retrieved_docs):
        print(f"\n  📄 Chunk {index + 1}:")
        print(f"  {doc.page_content}")

    # ── BUILD CONTEXT ─────────────────────────────────────────────────────────
    # Join all retrieved chunks into a single context string
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # ── CRAFT PROMPT ──────────────────────────────────────────────────────────
    # The instruction forces the LLM to stay grounded in the retrieved context.
    # The fallback message prevents confident hallucinations on out-of-scope questions.
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    # ── GENERATE ──────────────────────────────────────────────────────────────
    # Send the prompt to the LLM. response.content contains the generated text.
    response = llm.invoke(prompt)

    # ── OUTPUT ────────────────────────────────────────────────────────────────
    print("\n===== 🤖 ANSWER =====")
    print(response.content)

RAG Q&A System Ready. Type 'exit' to stop.


===== 📚 RETRIEVED CHUNKS =====

  📄 Chunk 1:
  Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

  📄 Chunk 2:
  Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.

===== 🤖 ANSWER =====
Employees are eligible for 6 months maternity leave.

👋 Goodbye!


---
## 🔁 Step 7b — (Alternative) Single-Shot Query — No Loop

Use this cell if you want to test a single question without entering the interactive loop above.

In [ ]:
# ── Change this question and re-run the cell to test different queries ────────
question = "What is the work from home policy?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

prompt = f"""You are a helpful assistant.
Answer ONLY using the provided context.
If the answer is not present, reply: "I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(f"❓ Question: {question}")
print(f"\n🤖 Answer: {response.content}")

❓ Question: What is the work from home policy?

🤖 Answer: Work from home is allowed twice a week.


---
## 🧪 Bonus — Load Existing Vector Store (Skip Re-indexing)

Once ChromaDB is persisted to disk, you don't need to re-index every time.  
This is the **production pattern** — index once, query many times.

In [20]:
# ── Load from existing persisted Chroma store (no re-embedding) ───────────────
# Use this after the first run — avoids duplicate indexing

vectorstore_loaded = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings   # Must use the SAME embedding model used during indexing
)

retriever_loaded = vectorstore_loaded.as_retriever(search_kwargs={"k": 2})

print(f"✅ Loaded existing store from disk")
print(f"📦 Vectors in store: {vectorstore_loaded._collection.count()}")

✅ Loaded existing store from disk
📦 Vectors in store: 14


---
## 🎯 Key Takeaways

| Concept | What to Remember |
|---|---|
| **Chunking** | Smaller & overlapping chunks = more precise retrieval |
| **Embeddings** | Text → numbers that capture meaning, not just keywords |
| **ChromaDB** | Local vector DB — index once, query forever |
| **`temperature=0`** | Always for RAG — you want determinism, not creativity |
| **Prompt guardrail** | "Answer ONLY from context" = the soul of a trustworthy RAG system |
| **Fallback message** | Prevents confident wrong answers on out-of-scope questions |

---

## 🚀 Next Steps to Explore

1. **Multi-document RAG** — Use `DirectoryLoader` to ingest hundreds of files
2. **Conversation memory** — Add `ConversationalRetrievalChain` for follow-up questions
3. **Reranking** — Add a cross-encoder reranker after retrieval for better precision
4. **Hybrid search** — Combine vector search + keyword (BM25) for edge cases where semantic search misses
5. **Evaluation** — Use `RAGAS` framework to measure faithfulness, context recall, and answer relevancy